In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CSV_PATH = r"C:\Users\HP-PC\Desktop\Project_X\Data\cleaned_booking.csv"
OUTPUT_DIR = r"C:\Users\HP-PC\Desktop\Project_X\Visuals"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
df["booking_hour"] = pd.to_datetime(
    df["time"], format="%H:%M:%S", errors="coerce"
).dt.hour

# -----------------------------------------------------------------------------
# CHART 1: Hourly Demand & Completion Rate
# -----------------------------------------------------------------------------
hourly = (
    df.groupby("booking_hour")
    .agg(
        total_requests=("booking_id", "count"),
        completed=(
            "booking_status",
            lambda s: (s == "Completed").sum(),
        ),
    )
    .reset_index()
)
hourly["completion_rate"] = (hourly["completed"] / hourly["total_requests"]) * 100

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(
    hourly["booking_hour"],
    hourly["total_requests"],
    color="#1f77b4",
    alpha=0.75,
    label="Ride Requests",
)
ax2.plot(
    hourly["booking_hour"],
    hourly["completion_rate"],
    color="#d62728",
    marker="o",
    linewidth=2,
    label="Completion Rate %",
)

ax1.set_xlabel("Hour of Day (0-23)")
ax1.set_ylabel("Total Requests", color="#1f77b4")
ax2.set_ylabel("Completion Rate (%)", color="#d62728")
ax2.set_ylim(50, 75)
plt.title("Hourly Demand Curve vs. Booking Completion Rate", fontsize=13, pad=12)
plt.xticks(range(0, 24))
plt.grid(True, linestyle="--", alpha=0.5)

chart1_path = os.path.join(OUTPUT_DIR, "01_hourly_demand.png")
plt.savefig(chart1_path, dpi=300)
plt.close()
print(f"Saved: {chart1_path}")

# -----------------------------------------------------------------------------
# CHART 2: Booking Value Distribution & Outlier Detection
# -----------------------------------------------------------------------------
completed = df[df["booking_status"] == "Completed"]["booking_value"]

plt.figure(figsize=(9, 4.5))
plt.hist(
    completed,
    bins=50,
    color="#2ca02c",
    edgecolor="black",
    alpha=0.7,
)
plt.axvline(
    completed.median(),
    color="blue",
    linestyle="--",
    linewidth=2,
    label=f"Median: {completed.median():.0f}",
)
plt.axvline(
    completed.mean(),
    color="red",
    linestyle="-",
    linewidth=2,
    label=f"Mean: {completed.mean():.0f}",
)

plt.title("Booking Value Distribution (Completed Rides)", fontsize=13, pad=12)
plt.xlabel("Booking Fare Value (INR)")
plt.ylabel("Trip Volume")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

chart2_path = os.path.join(OUTPUT_DIR, "02_fare_distribution.png")
plt.savefig(chart2_path, dpi=300)
plt.close()
print(f"Saved: {chart2_path}")

# -----------------------------------------------------------------------------
# CHART 3: Top 10 High-Failure Pickup Localities
# -----------------------------------------------------------------------------
loc_fail = (
    df.groupby("pickup_location")
    .agg(
        failed_trips=("booking_status", lambda s: (s != "Completed").sum()),
    )
    .sort_values(by="failed_trips", ascending=True)
    .tail(10)
)

plt.figure(figsize=(9, 5))
plt.barh(loc_fail.index, loc_fail["failed_trips"], color="#e377c2")
plt.title("Top 10 High-Failure Pickup Locations", fontsize=13, pad=12)
plt.xlabel("Failed / Unfulfilled Bookings")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)

chart3_path = os.path.join(OUTPUT_DIR, "03_failure_hotspots.png")
plt.savefig(chart3_path, dpi=300)
plt.close()
print(f"Saved: {chart3_path}")

Saved: C:\Users\HP-PC\Desktop\Project_X\Visuals\01_hourly_demand.png
Saved: C:\Users\HP-PC\Desktop\Project_X\Visuals\02_fare_distribution.png
Saved: C:\Users\HP-PC\Desktop\Project_X\Visuals\03_failure_hotspots.png
